# 07: Reproduce and understand the full training pipeline

[Course index](../README_en.md) | [中文版本](../notebooks/07_reproduce.ipynb)

**Goal:** connect the first six chapters into a MiniLLM pipeline that you can run, inspect, and explain on limited hardware.


## 1. Practical goal

Frontier models rely on clusters that most beginners cannot access. This project reduces the model to 17.23M parameters so you can personally experience tokenization, Base pretraining, SFT, preference alignment, and evaluation on an RTX 4060 8GB. Narrow instruction protocols and strict DPO are comparison experiments inside that pipeline, not grand claims the project must prove.


## 2. Pipeline order

```bash
python tokenizer/train_tokenizer.py
python scripts/prepare_pretrain_data.py
python train/pretrain.py --config configs/train_pretrain.yaml
python scripts/build_instruction_sft.py
python train/sft.py --config configs/train_sft.yaml
python scripts/build_strict_dpo.py
python train/dpo.py --config configs/train_instruction_dpo_v2.yaml
python eval/comprehensive_eval.py --output results/final_evaluation
```

Run tests and short smoke configurations before a long training job.


### Hands-on check

Load the formal model configuration and check required entry files.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "model").exists():
    ROOT = ROOT.parents[2]
sys.path.insert(0, str(ROOT))

from model.config import MiniLLMConfig
from model.gpt import MiniLLM

config = MiniLLMConfig.from_yaml(str(ROOT / "configs/model_config.yaml"))
model = MiniLLM(config)
required = [
    "tokenizer/minillm_tokenizer.json",
    "configs/train_pretrain.yaml",
    "configs/train_sft.yaml",
    "configs/train_instruction_dpo_v2.yaml",
    "eval/comprehensive_eval.py",
]
print(f"model parameters: {model.num_params():,}")
for relative in required:
    path = ROOT / relative
    print("OK" if path.exists() else "MISSING", relative)
assert model.num_params() == 17_232_768
assert all((ROOT / relative).exists() for relative in required)


## 3. Count updates correctly

One optimizer update processes

$$N_{tokens/update}=B_{micro}	imes N_{accum}	imes T.$$

Do not confuse progress-bar micro-steps with parameter updates.


### Hands-on check

Read the formal YAML and calculate token positions per optimizer update.


In [ ]:
import yaml

with (ROOT / "configs/train_pretrain.yaml").open(encoding="utf-8") as handle:
    train_config = yaml.safe_load(handle)
tokens_per_update = (
    train_config["batch_size"]
    * train_config["gradient_accumulation_steps"]
    * config.block_size
)
print("tokens per optimizer update:", tokens_per_update)
assert tokens_per_update == 32_768


## 4. Read comparisons as questions

Base→ContinuationSFT asks what extra story training does. ContinuationSFT→InstructionSFT isolates task structure. SFT→DPO-v1 tests reward overoptimization. SFT→DPO-v2 tests conservative strict pairs.


### Hands-on check

Calculate DPO-v2 changes relative to InstructionSFT from the report numbers.


In [ ]:
instruction_sft = {"ppl": 5.80, "sentence_exact": 0.615, "qa_em": 0.850}
dpo_v2 = {"ppl": 5.82, "sentence_exact": 0.715, "qa_em": 0.845}
delta = {key: dpo_v2[key] - instruction_sft[key] for key in instruction_sft}
print(delta)
assert round(delta["sentence_exact"], 3) == 0.100
assert round(delta["ppl"], 2) == 0.02


## 5. Reproducibility boundary

Code, configs, small JSONL data, reports, and tutorials are tracked. Large checkpoints, raw corpora, logs, and generated artifacts are local. Reproducing exact numbers from GitHub therefore requires base pretraining unless a checkpoint is distributed separately.


## 6. Responsible extensions

Prioritize multiple training seeds, stronger keyword supervision, blind human evaluation, then a 45M capacity comparison. Do not jump from TinyStories evidence to claims about general assistants.

- [Previous](06_evaluation.ipynb) · [Course index](../README_en.md)
- [Root README](../../../README.md)
- [Training and evaluation report](../../experiment_report.md)
- [Paper-to-code map](../../../papers/references_and_analysis.md)
